**Architecture for different projects:**

**Churn Prediction**

Raw Data
      ↓
Bronze
      ↓
Silver
      ↓
Gold
      ↓
Feature Engineering
      ↓
Machine Learning
      ↓
Evaluation
      ↓
Serving

**Support Ticket NLP with out embeddings**

Raw Tickets
      ↓
Bronze
      ↓
Silver
      ↓
Gold
      ↓
TF-IDF
      ↓
Machine Learning
      ↓
Evaluation

**RAG**

Raw Documents
      ↓
Bronze
      ↓
Silver
      ↓
Gold
      ↓
Embeddings
      ↓
Vector Search
      ↓
LLM

**Agentic AI**

Question
      ↓
Agent
      ↓
Tool Selection
      ↓
SQL
Prediction
Vector Search
API
      ↓
LLM

Pattern in the above every project follows the same high-level lifecycle:

Data
      ↓
Preparation
      ↓
Representation
      ↓
AI Model
      ↓
Evaluation
      ↓
Deployment/Application

The only thing that changes is how the data is represented:

- Structured ML → engineered numeric features
- Traditional NLP → TF-IDF features
- Modern NLP → embeddings
- Computer Vision → image tensors
- Multimodal AI → combinations of text, images, and structured data

This notebook Architecture:

create schema
      ↓    
Support Tickets data
      ↓    
 Bronze Table
      ↓
 Silver Table
      ↓
 Gold Table
      ↓
 TF-IDF Feature Engineering
      ↓
 Train Models
      ↓
 Write model_training_results


In [0]:
%sql
--- Step 1: create schema
CREATE SCHEMA IF NOT EXISTS dbw_agentic_ai_dev.support_ticket_ai;

In [0]:
# Step 2: import libraries
import random
from pyspark.sql import Row
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from mlflow.models.signature import infer_signature
import pandas as pd
import mlflow

In [0]:
# Step 3: Use templates to generate a larger balanced dataset.

random.seed(42)

ticket_templates = {
    "Technical": [
        "Internet is slow",
        "Wifi keeps disconnecting",
        "Connection drops every evening",
        "Router is not working",
        "Internet speed is very poor",
        "Service outage in my area",
        "Modem keeps restarting",
        "Video streaming keeps buffering",
        "Network connection is unstable",
        "Internet disconnects during meetings"
    ],
    "Login": [
        "Unable to login to my account",
        "Password reset is not working",
        "Account locked after many attempts",
        "Cannot access my customer portal",
        "Login page keeps failing",
        "Forgot my password",
        "Two factor authentication is not working",
        "My account access is blocked",
        "Unable to sign in",
        "Login error on mobile app"
    ],
    "Billing": [
        "My bill is too high",
        "I was charged incorrectly",
        "Unexpected charges on my invoice",
        "Refund not received",
        "Payment was processed twice",
        "Monthly bill amount is wrong",
        "Need explanation for extra fees",
        "Billing statement has incorrect amount",
        "Autopay charged the wrong card",
        "Invoice shows duplicate charge"
    ],
    "Cancellation": [
        "I want to cancel my service",
        "Please terminate my account",
        "I no longer need this service",
        "Cancel my subscription",
        "Close my account immediately",
        "I want to switch providers",
        "Stop my service next month",
        "I am leaving this provider",
        "Please disconnect my service",
        "I want to end my contract"
    ]
}


In [0]:
# Step 4: create bronze table

support_data = []
ticket_id = 1

for category, templates in ticket_templates.items():
    for i in range(25):
        text = random.choice(templates)
        support_data.append(
            Row(
                ticket_id=f"T{ticket_id:03d}",
                ticket_text=text,
                category=category
            )
        )
        ticket_id += 1

bronze_df = spark.createDataFrame(support_data)

bronze_df.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.bronze_support_tickets")

In [0]:
# Step 5: create silver table
silver_df = spark.sql("""
SELECT
    ticket_id,
    LOWER(TRIM(ticket_text)) AS cleaned_ticket_text,
    category
FROM dbw_agentic_ai_dev.support_ticket_ai.bronze_support_tickets
WHERE ticket_text IS NOT NULL
  AND category IS NOT NULL
""")

silver_df.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.silver_support_tickets")

In [0]:
# Step 6: create gold table
gold_df = spark.sql("""
SELECT
    ticket_id,
    cleaned_ticket_text,
    category
FROM dbw_agentic_ai_dev.support_ticket_ai.silver_support_tickets
""")

gold_df.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_features")

In [0]:
# Step 7: Read Gold table into pandas
df = spark.table( "dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_features").toPandas()

In [0]:
# Step 8: Train/test split

from sklearn.model_selection import train_test_split

X = df["cleaned_ticket_text"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Train Model
      ↓
Calculate Metrics
      ↓
Capture run_id
      ↓
Log Parameters
      ↓
Log Metrics
      ↓
Create Input Example
      ↓
Create Prediction Example
      ↓
Infer Signature
      ↓
Log Model
      ↓
Append Results

In [0]:
# Step 9: TF-IDF +  pipeline

user_name = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

experiment_path = f"/Users/{user_name}/support_ticket_experiment"

mlflow.set_experiment(experiment_path)

#print(experiment_path)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for model_name, classifier in models.items():

    #MLflow = Complete Experiment History.This stores everything about the experiment.
    #Start MLflow run
    with mlflow.start_run(run_name=model_name):
         
        pipeline = Pipeline(
            steps=[
                ("tfidf", TfidfVectorizer(stop_words="english")),
                ("classifier", classifier)
            ]
        )

        #Train model
        pipeline.fit(X_train, y_train)

        #Predict
        y_pred = pipeline.predict(X_test)

        #Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision_macro = precision_score(y_test, y_pred, average="macro")
        recall_macro = recall_score(y_test, y_pred, average="macro")
        f1_macro = f1_score(y_test, y_pred, average="macro")
       
        vocab_size = len(pipeline.named_steps["tfidf"].vocabulary_)

        #log parameters
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("vectorizer", "TF-IDF")
        mlflow.log_param("stop_words", "english")

        #log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision_macro", precision_macro)
        mlflow.log_metric("recall_macro", recall_macro)
        mlflow.log_metric("f1_macro", f1_macro)  
        mlflow.log_metric("vocabulary_size", vocab_size)

        # Create sample input as DataFrame
        input_example = X_train.head(5).to_frame(name="cleaned_ticket_text")

        # Get sample prediction output
        prediction_example = pipeline.predict(input_example["cleaned_ticket_text"])

        # Infer model signature
        signature = infer_signature(
            input_example,
            prediction_example
        )

        # log the model
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="support_ticket_model",
            signature=signature,
            input_example=input_example
        )
        
        #Capture the run_id
        run_id = mlflow.active_run().info.run_id        

        #append results to the delta table. Delta Table = Business Summary
        results.append({
            "run_id": run_id,
            "model": model_name,
            "vectorizer": "TF-IDF",
            "vocabulary_size": vocab_size,
            "accuracy": accuracy,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro
        })

results_df = pd.DataFrame(results)

results_spark = spark.createDataFrame(results_df)

results_spark.write.mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.model_training_results")